# Problem without Pydantic AI 

In [8]:
def add_student_data(name:str,age:int):
  if type(name)==str and type(age)==int:
    if age>=0:
      print(f"Student Name: {name}, Age: {age}")
    else:
      raise ValueError("Age must be a non-negative integer.")
  else:
    raise TypeError("Invalid data type for name or age. Name must be a string and age must be an integer.")

def update_student_data(name:str,age:int):
  if type(name)==str and type(age)==int:
    if age>=0:
      print(f"Updated Student Name: {name}, Updated Age: {age}")
    else:
      raise ValueError("Age must be a non-negative integer.")
  else:
    raise TypeError("Invalid data type for name or age. Name must be a string and age must be an integer.")

In [9]:
add_student_data("John Doe", 20)
# Problem - Manually handling the validation of data types and values can lead to repetitive code and potential errors.

Student Name: John Doe, Age: 20


In [26]:
%pip install pydantic
%pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
    --------------------------------------- 0.0/1.8 MB 326.8 kB/s eta 0:00:06
   - -------------------------------------- 0.1/1.8 MB 290.5 kB/s eta 0:00:07
   - -------------------------------------- 0.1/1.8 MB 357.2 kB/s eta 0:00:05
   -- ------------------------------------- 0.1/1.8 MB 504.4 kB/s eta 0:00:04
   -- ------------------------------------- 0.1/1.8 MB 481.4 kB/s eta 0:00:04
   --- ------------------------------------ 0.2/1.8 MB 551.6 kB/s eta 0:00:03
   ---- ----------------------------------- 0.2/1.8 MB 562.0 kB/s eta 0:00:03
   ---- --------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [29]:
%pip install email-validator


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Data Validator


In [45]:
from pydantic import BaseModel, EmailStr,Field, field_validator,model_validator,computed_field
# field is used for custom behavior
from typing import List, Optional, Dict, Annotated

class Student(BaseModel):
    name: Annotated[str, Field(max_length=50, description="Name of the student",example=["John Doe","Jane Smith"])]
    age: Annotated[int, Field(gt=0, strict=True, description="Age of the student, must be a positive integer")]
    hobby: List[str]
    contact: Dict[str, str]
    allergies: Optional[List[str]] = Field(default=None, max_length=5, description="List of allergies, optional field with a maximum length of 5")
    email: EmailStr

student_data = {
    "name": "John Doe",
    "age": 20,  # will be auto-converted to int
    "hobby": ["reading", "swimming"],
    "contact": {"email": "john.doe@example.com", "phone": "4557551"},
    "email": "john.doe@example.com"
}

student = Student(**student_data)
print(student)


name='John Doe' age=20 hobby=['reading', 'swimming'] contact={'email': 'john.doe@example.com', 'phone': '4557551'} allergies=None email='john.doe@example.com'


C:\Users\impav\AppData\Local\Temp\ipykernel_20564\3654109672.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  name: Annotated[str, Field(max_length=50, description="Name of the student",example=["John Doe","Jane Smith"])]


In [20]:
student.age,student.name,student.hobby,student.contact,student.allergies

(20,
 'John Doe',
 ['reading', 'swimming'],
 {'email': 'john.doe@example.com', 'phone': '4557551'},
 None)

# Field Validator

In [43]:
class Patient(BaseModel):
  name:str
  email:EmailStr
  age:int
  weight:float
  married:bool
  allergies:List[str]
  contact_details:Dict[str,str]

  @field_validator('email')
  @classmethod
  def validate_email(cls, value):
    valid_domains=["icici.com","sbi.com","hdfc.com"]
    domain_name=value.split('@')[-1]

    if domain_name not in valid_domains:
      raise ValueError(f"Email domain must be one of {valid_domains}")

    return value

def add_patient_data(patient:Patient):
  print(f"Patient Name: {patient.name}, Email: {patient.email}, Age: {patient.age}, Weight: {patient.weight}, Married: {patient.married}, Allergies: {patient.allergies}, Contact Details: {patient.contact_details}")

patient_data ={"name":"Harsh","email": "harsh@icici.com","age": 25,"weight": 70.5,"married": False,"allergies": ["pollen", "dust"],"contact_details": {"phone": "1234567890", "address": "123 Main St"}}

patient1=Patient(**patient_data)
add_patient_data(patient1)

Patient Name: Harsh, Email: harsh@icici.com, Age: 25, Weight: 70.5, Married: False, Allergies: ['pollen', 'dust'], Contact Details: {'phone': '1234567890', 'address': '123 Main St'}


# Model Validator and Compute validator

In [49]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator, computed_field
from typing import List, Dict

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]
    height: float

    @model_validator(mode="after")
    def validate_emergency_contact(self):
        if self.age > 60 and "emergency" not in self.contact_details:
            raise ValueError("Emergency contact is required for patients above 60 years old.")
        return self

    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / (self.height ** 2), 2)


def add_patient_data(patient: Patient):
    print(
        f"Patient Name: {patient.name}, Email: {patient.email}, Age: {patient.age}, "
        f"Weight: {patient.weight}, Married: {patient.married}, Allergies: {patient.allergies}, "
        f"Contact Details: {patient.contact_details}, BMI: {patient.bmi}"
    )


patient_data = {
    "name": "Harsh",
    "email": "harsh@icici.com",
    "age": 25,
    "weight": 70.5,
    "married": False,
    "allergies": ["pollen", "dust"],
    "contact_details": {"phone": "1234567890", "address": "123 Main St"},
    "height": 1.75,
}

patient1 = Patient(**patient_data)
add_patient_data(patient1)


Patient Name: Harsh, Email: harsh@icici.com, Age: 25, Weight: 70.5, Married: False, Allergies: ['pollen', 'dust'], Contact Details: {'phone': '1234567890', 'address': '123 Main St'}, BMI: 23.02


# Serialization

In [ ]:


# serialize to Python dict
student_dict = student.model_dump()
print("dict:", student_dict)

# serialize to JSON
student_json = student.model_dump_json()
print("json:", student_json)

# deserialize back from dict
student_from_dict = Student(**student_dict)
print("from dict:", student_from_dict)

# deserialize back from JSON
student_from_json = Student.model_validate_json(student_json)
print("from json:", student_from_json)


dict: {'name': 'John Doe', 'age': 20, 'hobby': ['reading', 'swimming'], 'contact': {'email': 'john.doe@example.com', 'phone': '4557551'}, 'allergies': None, 'email': 'john.doe@example.com'}
json: {"name":"John Doe","age":20,"hobby":["reading","swimming"],"contact":{"email":"john.doe@example.com","phone":"4557551"},"allergies":null,"email":"john.doe@example.com"}
from dict: name='John Doe' age=20 hobby=['reading', 'swimming'] contact={'email': 'john.doe@example.com', 'phone': '4557551'} allergies=None email='john.doe@example.com'
from json: name='John Doe' age=20 hobby=['reading', 'swimming'] contact={'email': 'john.doe@example.com', 'phone': '4557551'} allergies=None email='john.doe@example.com'


# Pydantic in AI Applications: Core Benefits

---

### ⚙️ Core Data & Structure Control
* **JSON Schema Generation:** Automatically creates strict JSON schemas from Python code to guide LLM outputs.
* **Type Safety:** Converts raw AI string outputs into concrete Python types (like `int`, `datetime`, or custom objects).
* **Deep Validation:** Enforces business logic rules (e.g., checking if a parsed string is a valid email or positive number).
* **Graceful Failures:** Catches messy or incomplete AI responses before they reach production code and crash the app.

### 🤖 AI Agent & Tool Execution
* **Function Calling:** Translates Python type hints and docstrings into clean API specifications that agents can read and execute.
* **Error Self-Correction:** Feeds validation error messages back to the LLM so the agent can automatically rewrite and fix its own bad output.
* **Context Injection:** Passes runtime dependencies (like database connections or API keys) securely into agent tools.

### Production & Development Efficiency
* **Framework Standard:** Serves as the native foundation for major AI orchestration tools like LangChain, LlamaIndex, and OpenAI's structured outputs.
* **Built-in Observability:** Integrates natively with tools like Pydantic Logfire to monitor agent steps, track latency, and debug prompts.
* **IDE Support:** Provides instant auto-complete and linting for AI data structures, accelerating development speed.
